# Qwen3 Extractive Causal LM 4h

Notebook n?y train m?t Qwen3-1.7B LoRA c?u h?nh m?nh h?n cho dataset t?m t?t near-extractive, r?i evaluate tr?c ti?p tr?n `test-00000-of-00001.parquet`.

M?c ti?u th?i gian tr?n Kaggle T4x2: train + test kho?ng d??i 4 gi?. N?u ETA v??t 4 gi?, gi?m `TRAIN_MAX_STEPS` xu?ng 400 ho?c ??t `MAX_TEST_SAMPLES = 500`.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time
from zipfile import ZipFile

PROJECT_NAME = 'pretrained-summarization'
REPO_URL = 'https://github.com/Anhnguyen0812/pretrained-summarization.git'
REFRESH_REPO = True
WORKING = Path('/kaggle/working')
WORKING_REPO = WORKING / PROJECT_NAME
OUTPUT_ROOT = WORKING / 'qwen3_extractive_4h_outputs'
REPORT_DIR = OUTPUT_ROOT / '_report'
DATA_DIR = Path('/kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization')
TRAIN_FILE = DATA_DIR / 'train-00000-of-00001.parquet'
VALID_FILE = DATA_DIR / 'valid-00000-of-00001.parquet'
TEST_FILE = DATA_DIR / 'test-00000-of-00001.parquet'

RUN_NAME = 'qwen3_extractive_prompt_r16_500s_4h'
TRAIN_MAX_STEPS = 500
TRAIN_MAX_SAMPLES = 9000
VALID_MAX_SAMPLES = 120
MAX_TEST_SAMPLES = None  # None = full 1344 test samples; set 500 if need faster run
TEST_BATCH_SIZE = 4
TEST_MAX_NEW_TOKENS = 96
TEST_NUM_BEAMS = 1
OVERWRITE_RUN = False
OVERWRITE_TEST = True

os.chdir(WORKING)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, check=True):
    print('CMD:', cmd, flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    lines = []
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    code = process.wait()
    if check and code != 0:
        raise RuntimeError(f'Command failed with exit code {code}: {cmd}\nLast lines:\n{"".join(lines[-100:])}')
    return code

def is_repo(path):
    return (path / 'pyproject.toml').exists() and (path / 'src' / 'vn_summarization').exists()

if REFRESH_REPO and WORKING_REPO.exists():
    shutil.rmtree(WORKING_REPO)
if not is_repo(WORKING_REPO):
    run(f'git clone --depth 1 {REPO_URL} {WORKING_REPO}', cwd=WORKING)
else:
    run('git pull --ff-only', cwd=WORKING_REPO, check=False)
repo = WORKING_REPO
run('git log --oneline -1', cwd=repo)


In [ ]:
os.chdir(repo)
run(f'{sys.executable} -m pip install -q --upgrade pip', cwd=repo)
run(f'{sys.executable} -m pip install -q -e .', cwd=repo)
run(f'{sys.executable} -m pip install -q --upgrade "transformers>=4.51.0,<5" "tokenizers>=0.22.0,<=0.23.0"', cwd=repo)
run(f'{sys.executable} -m pip check', cwd=repo, check=False)
run(f'{sys.executable} -m pip show transformers tokenizers peft accelerate | sed -n "/Name: /p;/Version: /p"', cwd=repo, check=False)
run(f"{sys.executable} -c 'import tokenizers, transformers; print(\"TRANSFORMERS\", transformers.__version__); print(\"TOKENIZERS\", tokenizers.__version__)'", cwd=repo)

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
run('nvidia-smi', check=False)

for path in [TRAIN_FILE, VALID_FILE, TEST_FILE]:
    print(path, path.exists())
if not TRAIN_FILE.exists() or not VALID_FILE.exists() or not TEST_FILE.exists():
    raise FileNotFoundError('Attach Kaggle dataset anhnguyen0812/nlp-vietnamese-sumarization first.')

import torch
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
print('NUM_GPUS:', NUM_GPUS)


## Config

K? thu?t b?t trong run n?y:

- prompt extractive, ?u ti?n copy c?m t? quan tr?ng v? dataset c? ROUGE/copy n-gram cao
- loss mask ph?n prompt ?? c? s?n trong `causal_data.py`, ch? h?c ph?n summary
- LoRA r16 tr?n attention + MLP
- context 1024 ?? gi? nhi?u article h?n
- train nhi?u m?u h?n run 300 steps, nh?ng gi?i h?n 500 steps ?? gi? budget th?i gian
- test greedy, `max_new_tokens=96`, batch 4 ?? ch?y nhanh h?n beam search


In [ ]:
import yaml

GENERATED_CONFIG_DIR = repo / 'configs' / '_generated_4h'
GENERATED_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = GENERATED_CONFIG_DIR / f'{RUN_NAME}.yaml'
RUN_DIR = OUTPUT_ROOT / RUN_NAME
TEST_OUT_DIR = OUTPUT_ROOT / f'{RUN_NAME}_test'

EXTRACTIVE_PROMPT = (
    "B\u1ea1n l\u00e0 h\u1ec7 th\u1ed1ng t\u00f3m t\u1eaft tin t\u1ee9c ti\u1ebfng Vi\u1ec7t. "
    "Vi\u1ebft duy nh\u1ea5t m\u1ed9t \u0111o\u1ea1n t\u00f3m t\u1eaft 80-130 t\u1eeb. "
    "Ch\u1ec9 d\u00f9ng th\u00f4ng tin trong v\u0103n b\u1ea3n. "
    "\u01afu ti\u00ean gi\u1eef l\u1ea1i c\u00e1c c\u1ee5m t\u1eeb quan tr\u1ecdng xu\u1ea5t hi\u1ec7n trong v\u0103n b\u1ea3n, "
    "t\u00ean ri\u00eang, \u0111\u1ecba danh, t\u1ed5 ch\u1ee9c, s\u1ed1 li\u1ec7u v\u00e0 th\u1eddi gian. "
    "Kh\u00f4ng suy lu\u1eadn, kh\u00f4ng gi\u1ea3i th\u00edch, kh\u00f4ng d\u00f9ng th\u1ebb <think>, kh\u00f4ng g\u1ea1ch \u0111\u1ea7u d\u00f2ng.\n\n"
    "V\u0103n b\u1ea3n:\n{article}\n\nT\u00f3m t\u1eaft:\n"
)

with (repo / 'configs' / 'qwen3_1_7b_lora.yaml').open('r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['data'].update({
    'train_file': str(TRAIN_FILE),
    'valid_file': str(VALID_FILE),
    'max_source_length': 1024,
    'max_target_length': 160,
    'max_length': 1184,
    'preprocessing_num_proc': 2,
    'max_train_samples': TRAIN_MAX_SAMPLES,
    'max_eval_samples': VALID_MAX_SAMPLES,
    'prompt_template': EXTRACTIVE_PROMPT,
})
cfg['training'].update({
    'output_dir': str(RUN_DIR),
    'overwrite_output_dir': OVERWRITE_RUN,
    'precision': 'fp16',
    'per_device_train_batch_size': 1,
    'per_device_eval_batch_size': 2,
    'gradient_accumulation_steps': 8,
    'num_train_epochs': 1,
    'max_steps': TRAIN_MAX_STEPS,
    'learning_rate': 8e-5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.04,
    'lr_scheduler_type': 'cosine',
    'optim': 'adamw_torch',
    'gradient_checkpointing': True,
    'strategy': 'steps',
    'save_strategy': 'steps',
    'eval_steps': max(250, TRAIN_MAX_STEPS // 2),
    'save_steps': max(250, TRAIN_MAX_STEPS // 2),
    'logging_steps': 25,
    'save_total_limit': 2,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'eval_loss',
    'greater_is_better': False,
    'ddp_find_unused_parameters': False,
    'save_safetensors': True,
})
cfg['generation'].update({
    'max_new_tokens': 128,
    'num_beams': 1,
    'do_sample': False,
    'no_repeat_ngram_size': 3,
    'repetition_penalty': 1.05,
})
cfg['lora'].update({
    'enabled': True,
    'r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
})

with CONFIG_PATH.open('w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)

print('CONFIG_PATH:', CONFIG_PATH)
print(CONFIG_PATH.read_text(encoding='utf-8'))


In [ ]:
def latest_checkpoint(run_dir):
    checkpoints = sorted(
        run_dir.glob('checkpoint-*'),
        key=lambda p: int(p.name.split('-')[-1]) if p.name.split('-')[-1].isdigit() else -1,
    )
    return checkpoints[-1] if checkpoints else None

def launch_train(config_path):
    rel_config = Path(config_path).relative_to(repo).as_posix()
    train_cmd = f'-m vn_summarization.train_causal_lm --config {rel_config}'
    if NUM_GPUS >= 2:
        cmd = f'{sys.executable} -m accelerate.commands.launch --multi_gpu --num_processes {NUM_GPUS} --num_machines 1 --mixed_precision fp16 --dynamo_backend no {train_cmd}'
    else:
        cmd = f'{sys.executable} -u {train_cmd}'
    return run(cmd, cwd=repo)

start = time.time()
if (RUN_DIR / 'best' / 'adapter_config.json').exists() and not OVERWRITE_RUN:
    print('SKIP train, adapter exists:', RUN_DIR / 'best')
else:
    if RUN_DIR.exists() and OVERWRITE_RUN:
        shutil.rmtree(RUN_DIR)
    if RUN_DIR.exists() and not OVERWRITE_RUN:
        ckpt = latest_checkpoint(RUN_DIR)
        if ckpt:
            print('RESUME checkpoint:', ckpt)
            with CONFIG_PATH.open('r', encoding='utf-8') as f:
                resume_cfg = yaml.safe_load(f)
            resume_cfg.setdefault('training', {})['resume_from_checkpoint'] = str(ckpt)
            with CONFIG_PATH.open('w', encoding='utf-8') as f:
                yaml.safe_dump(resume_cfg, f, allow_unicode=True, sort_keys=False)
    launch_train(CONFIG_PATH)
print('TRAIN elapsed_hours:', round((time.time() - start) / 3600, 3))


In [ ]:
if TEST_OUT_DIR.exists() and OVERWRITE_TEST:
    shutil.rmtree(TEST_OUT_DIR)
TEST_OUT_DIR.mkdir(parents=True, exist_ok=True)

args = (
    f'--runs_root {OUTPUT_ROOT} '
    f'--run_glob {RUN_NAME} '
    f'--test_file {TEST_FILE} '
    f'--out_dir {TEST_OUT_DIR} '
    f'--eval_batch_size {TEST_BATCH_SIZE} '
    f'--generation_max_new_tokens {TEST_MAX_NEW_TOKENS} '
    f'--generation_num_beams {TEST_NUM_BEAMS}'
)
if MAX_TEST_SAMPLES is not None:
    args += f' --max_test_samples {MAX_TEST_SAMPLES}'

test_start = time.time()
run(f'{sys.executable} -u -m vn_summarization.evaluate_runs_on_test {args}', cwd=repo)
print('TEST elapsed_hours:', round((time.time() - test_start) / 3600, 3))
print('TOTAL elapsed_hours:', round((time.time() - start) / 3600, 3))


In [ ]:
import csv

def load_json(path):
    if not path.exists():
        return {}
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)

result_csv = TEST_OUT_DIR / 'test_results.csv'
rows = []
if result_csv.exists():
    with result_csv.open('r', encoding='utf-8', newline='') as f:
        rows = list(csv.DictReader(f))

train_metrics = load_json(RUN_DIR / 'train_results.json')
eval_metrics = load_json(RUN_DIR / 'eval_results.json')
best = load_json(TEST_OUT_DIR / 'best_test_run.json')
summary = {
    'run': RUN_NAME,
    'train_max_steps': TRAIN_MAX_STEPS,
    'train_max_samples': TRAIN_MAX_SAMPLES,
    'valid_max_samples': VALID_MAX_SAMPLES,
    'test_max_samples': MAX_TEST_SAMPLES or 'all',
    'test_batch_size': TEST_BATCH_SIZE,
    'test_max_new_tokens': TEST_MAX_NEW_TOKENS,
    'train_runtime': train_metrics.get('train_runtime', ''),
    'eval_loss': eval_metrics.get('eval_loss', ''),
    'test_rouge1': best.get('rouge1', ''),
    'test_rouge2': best.get('rouge2', ''),
    'test_rougeL': best.get('rougeL', ''),
    'test_gen_len': best.get('gen_len', ''),
    'metrics_file': best.get('metrics_file', ''),
    'predictions_file': best.get('predictions_file', ''),
}

summary_path = REPORT_DIR / 'qwen3_extractive_4h_summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

md = ['# Qwen3 Extractive 4h Result', '']
md.append('| metric | value |')
md.append('| --- | --- |')
for key, value in summary.items():
    md.append(f'| {key} | {value} |')
md.append('')
if (TEST_OUT_DIR / 'test_results.md').exists():
    md.append((TEST_OUT_DIR / 'test_results.md').read_text(encoding='utf-8'))
report_md = REPORT_DIR / 'qwen3_extractive_4h_summary.md'
report_md.write_text('\n'.join(md), encoding='utf-8')
print(report_md.read_text(encoding='utf-8'))


In [ ]:
zip_path = WORKING / 'qwen3_extractive_4h_results.zip'
if zip_path.exists():
    zip_path.unlink()
keep_suffixes = {'.json', '.jsonl', '.csv', '.md', '.txt', '.model', '.safetensors', '.jinja'}

def keep_file(path):
    if not path.is_file() or path.suffix not in keep_suffixes:
        return False
    if any(part.startswith('checkpoint-') for part in path.parts):
        return False
    return True

files = [p for p in OUTPUT_ROOT.rglob('*') if keep_file(p)]
with ZipFile(zip_path, 'w') as zf:
    for file in files:
        zf.write(file, file.relative_to(WORKING).as_posix())
print('ZIP:', zip_path)
for file in sorted(files):
    print(file)
